# 1. Data Understanding

This notebook uses the City of Melbourne open-data dataset, **Landmarks and places of interest including schools, theatres, health services, sports...**. It represents listed places of interest and their categories and coordinates; it does not represent confirmed live events.

The dataset is relevant to AgeTogether because it can support local place discovery for older adults living independently in Melbourne. However, it does not provide live event availability, booking information, prices, opening hours, or accessibility details. These details must not be inferred from this dataset.

In [ ]:
from pathlib import Path

import pandas as pd

raw_dataset_filename = "landmarks-and-places-of-interest-including-schools-theatres-health-services-spor.csv"
csv_path = Path("../data/raw") / raw_dataset_filename
if not csv_path.exists():
    csv_path = Path("data/raw") / raw_dataset_filename

df = pd.read_csv(csv_path)

df.head()


## 1.1 Dataset Overview

This section shows the size, column names, and data types of the raw dataset. A data type describes the kind of value stored in a column, such as text or number.

In [ ]:
overview = pd.DataFrame({
    "Measure": ["Number of rows", "Number of columns", "Column names"],
    "Value": [df.shape[0], df.shape[1], ", ".join(df.columns)]
})

print(overview.to_string(index=False))

data_types = df.dtypes.rename("Data type").reset_index()
data_types.columns = ["Column", "Data type"]
print(data_types.to_string(index=False))

## 1.2 Data Quality Checks

These checks look for missing values, exact duplicate rows, repeated place names, and the number of categories. An exact duplicate row means every value in one row is identical to another row.

In [ ]:
missing_values = df.isna().sum().rename("Missing values").reset_index()
missing_values.columns = ["Column", "Missing values"]
print(missing_values.to_string(index=False))

repeated_feature_names = (
    df.loc[df["Feature Name"].duplicated(keep=False), "Feature Name"]
    .value_counts()
    .rename_axis("Feature Name")
    .reset_index(name="Number of records")
)

quality_summary = pd.DataFrame({
    "Check": [
        "Exact duplicate rows",
        "Duplicated Feature Name entries after the first occurrence",
        "Distinct Feature Names that are repeated",
        "Unique Theme values",
        "Unique Sub Theme values"
    ],
    "Result": [
        df.duplicated().sum(),
        df["Feature Name"].duplicated().sum(),
        repeated_feature_names.shape[0],
        df["Theme"].nunique(),
        df["Sub Theme"].nunique()
    ]
})

print(quality_summary.to_string(index=False))
print(repeated_feature_names.to_string(index=False))

## 1.3 Coordinate Check

The `Co-ordinates` field stores latitude and longitude together as text. This check validates temporary numeric coordinate values without changing the raw dataset. Latitude must be between -90 and 90, and longitude must be between -180 and 180.

In [ ]:
coordinates = df["Co-ordinates"].str.split(",", n=1, expand=True)
latitude_check = pd.to_numeric(coordinates[0].str.strip(), errors="coerce")
longitude_check = pd.to_numeric(coordinates[1].str.strip(), errors="coerce")

coordinates_parsed = pd.DataFrame({"latitude": latitude_check, "longitude": longitude_check}).notna().all(axis=1)
valid_geographic_range = (
    latitude_check.between(-90, 90)
    & longitude_check.between(-180, 180)
)

coordinate_summary = pd.DataFrame({
    "Check": [
        "Coordinates parsed successfully",
        "Minimum latitude",
        "Maximum latitude",
        "Minimum longitude",
        "Maximum longitude",
        "All coordinates within valid geographic ranges"
    ],
    "Result": [
        coordinates_parsed.sum(),
        latitude_check.min(),
        latitude_check.max(),
        longitude_check.min(),
        longitude_check.max(),
        valid_geographic_range.all()
    ]
})

print(coordinate_summary.to_string(index=False))

## 1.4 Initial Data Quality Observations

- The dataset has 242 rows and 4 original columns, with no missing values in those original columns.
- There are no exact duplicate rows. Four Feature Names are repeated, producing 9 duplicated Feature Name entries after their first occurrence; these records should be reviewed in a later step before use.
- All 242 coordinate values were parsed successfully, and all fall within valid geographic ranges.
- The dataset contains 16 Theme values and 49 Sub Theme values. These categories can support later, transparent relevance decisions, but no filtering or ranking is performed in this notebook section.
- For AgeTogether, this is place-of-interest data only. It cannot confirm live events, availability, bookings, prices, opening hours, or accessibility details.

# 2. Data Cleaning and Preparation

This section prepares an in-memory processed copy for later analysis. The original raw CSV file and its original columns are preserved. No filtering, ranking, recommendation, or frontend integration is performed here.

## 2.1 Coordinate Transformation

A processed copy is created so the original `Co-ordinates` text field is retained. Latitude and longitude are split into separate numeric columns. Values that cannot be converted are kept as missing values for review; they are not discarded.

In [ ]:
prepared_df = df.copy()
coordinate_parts = prepared_df["Co-ordinates"].str.split(",", n=1, expand=True)
prepared_df["latitude"] = pd.to_numeric(coordinate_parts[0].str.strip(), errors="coerce")
prepared_df["longitude"] = pd.to_numeric(coordinate_parts[1].str.strip(), errors="coerce")

coordinate_parse_success = prepared_df[["latitude", "longitude"]].notna().all(axis=1)
coordinate_parse_failures = prepared_df.loc[
    ~coordinate_parse_success,
    ["Co-ordinates", "latitude", "longitude"]
]

coordinate_transformation_summary = pd.DataFrame({
    "Check": ["Coordinates parsed successfully", "Coordinate parsing failures"],
    "Result": [coordinate_parse_success.sum(), (~coordinate_parse_success).sum()]
})

print(coordinate_transformation_summary.to_string(index=False))
if coordinate_parse_failures.empty:
    print("No coordinate parsing failures were found.")
else:
    print(coordinate_parse_failures.to_string(index=False))

## 2.2 Duplicate Feature Name Review

A repeated Feature Name is not automatically a duplicate record. The review below compares its category and coordinate values before any future cleaning rule is considered.

In [ ]:
duplicate_feature_review = prepared_df.loc[
    prepared_df["Feature Name"].duplicated(keep=False),
    ["Feature Name", "Theme", "Sub Theme", "latitude", "longitude"]
].sort_values(["Feature Name", "latitude", "longitude"])

duplicate_group_summary = (
    duplicate_feature_review.groupby("Feature Name", as_index=False)
    .agg(
        records=("Feature Name", "size"),
        unique_themes=("Theme", "nunique"),
        unique_sub_themes=("Sub Theme", "nunique")
    )
)
unique_coordinate_counts = (
    duplicate_feature_review.drop_duplicates(["Feature Name", "latitude", "longitude"])
    .groupby("Feature Name")
    .size()
    .rename("unique_coordinate_pairs")
    .reset_index()
)
duplicate_group_summary = duplicate_group_summary.merge(
    unique_coordinate_counts,
    on="Feature Name",
    how="left"
)

print(duplicate_feature_review.to_string(index=False))
print(duplicate_group_summary.to_string(index=False))

**Review result:** Each of the four repeated Feature Names has different coordinate pairs across its records, while its Theme and Sub Theme values are the same within that name. These records should be retained. A future rule should only consider a record for possible deduplication when the Feature Name and coordinate pair are both identical, followed by manual review because this dataset has no unique record identifier.

## 2.3 Categorical Consistency Review

This review lists the source category values without changing them. It also checks whether values become identical after converting only their capitalisation to a common form.

In [ ]:
theme_values = pd.DataFrame({"Theme": sorted(prepared_df["Theme"].unique())})
sub_theme_values = pd.DataFrame({"Sub Theme": sorted(prepared_df["Sub Theme"].unique())})

def find_case_only_collisions(series):
    grouped_values = series.groupby(series.str.strip().str.casefold()).unique()
    return {key: list(values) for key, values in grouped_values.items() if len(values) > 1}

theme_case_collisions = find_case_only_collisions(prepared_df["Theme"])
sub_theme_case_collisions = find_case_only_collisions(prepared_df["Sub Theme"])

display_review = pd.DataFrame({
    "Field": ["Theme", "Sub Theme"],
    "Finding": [
        "Place Of Assembly and Place of Worship use different capitalisation styles for the word of; review this only for user-facing display.",
        "No case-only duplicate values were found."
    ]
})

print(theme_values.to_string(index=False))
print(sub_theme_values.to_string(index=False))
print("Theme case-only collisions:", theme_case_collisions)
print("Sub Theme case-only collisions:", sub_theme_case_collisions)
print(display_review.to_string(index=False))

## 2.4 Cleaning Decisions

1. A processed in-memory copy named `prepared_df` was created. The original `Co-ordinates` value was retained, and separate numeric `latitude` and `longitude` columns were added.
2. No records were removed. Repeated Feature Names have different coordinate pairs, so a name match alone is not sufficient evidence that records are duplicates.
3. Repeated Feature Names require continued review if a later task needs record-level deduplication. The Theme labels `Place Of Assembly` and `Place of Worship` also use different capitalisation styles and may need a separate display-label rule; the source values remain unchanged.
4. Keeping the original coordinate text and adding numeric coordinate columns to a processed copy is a safe transformation for later analysis. No category values have been changed.
5. The source does not provide live event availability, booking information, prices, opening hours, accessibility details, or confirmed availability. This information must not be invented.

# 3. Data Dictionary and Data Provenance

## 3.1 Data Dictionary

A data dictionary explains the fields in a dataset, including what is directly available and how a field may be used without adding unsupported information.

| Field name | Data type | Meaning / interpretation | Potential use in AgeTogether | Important limitation |
|---|---|---|---|---|
| Theme | object (text) | Source category label for the place record. | Support later organisation of places by source category. | The CSV does not provide a formal definition for each category or confirm that a place is suitable for a particular activity. |
| Sub Theme | object (text) | Source sub-category label for the place record. | Support more detailed organisation of places by source sub-category. | The CSV does not provide a formal definition for each sub-category or any live activity information. |
| Feature Name | object (text) | Name recorded for the place of interest. | Display or identify a listed place. | It is not a unique record identifier: the same name can appear at different coordinates. |
| Co-ordinates | object (text) | Original source text containing a latitude and longitude value separated by a comma. | Preserved source location value for traceability. | It is text rather than separate numeric location fields. |
| latitude | float64 (numeric, derived) | Derived by splitting `Co-ordinates` and converting the first value to a number. | Support later location-based analysis of listed places. | It is derived from the source coordinate text; it does not provide accessibility, distance from a user, or live availability. |
| longitude | float64 (numeric, derived) | Derived by splitting `Co-ordinates` and converting the second value to a number. | Support later location-based analysis of listed places. | It is derived from the source coordinate text; it does not provide accessibility, distance from a user, or live availability. |

## 3.2 Data Provenance

Data provenance records where data came from and how it has been handled. The following table distinguishes known project information from unavailable metadata.

| Metadata item | Verified information |
|---|---|
| Dataset name | Landmarks and places of interest, including schools, theatres, health services, sports facilities, places of worship, galleries and museums. |
| Data provider/source | City of Melbourne |
| Official dataset identifier | landmarks-and-places-of-interest-including-schools-theatres-health-services-spor |
| Local file format | CSV |
| Access method | The team downloaded a local CSV copy. |
| Official source URL | https://data.melbourne.vic.gov.au/explore/dataset/landmarks-and-places-of-interest-including-schools-theatres-health-services-spor/information/ |
| Local acquisition date | 1 September 2026. This is the team's local download date, not an official dataset update date. |
| Licence | CC BY |
| Official creation date | 30 April 2014 |
| Official modified date | 12 March 2021 |
| Official last data processing date | 13 November 2022 |
| Original dataset structure | 242 rows × 4 columns: Theme, Sub Theme, Feature Name, Co-ordinates. |
| Current processed dataframe structure | `prepared_df` contains 242 rows × 6 columns, retaining the four original fields and adding derived numeric latitude and longitude fields. |
| Raw source file modified | No. The notebook reads the CSV and creates `prepared_df` in memory; it does not write to the raw CSV. |
| Data freshness / update date | The official modified date is 12 March 2021 and the official last data processing date is 13 November 2022. The 1 September 2026 local acquisition date does not mean the dataset was updated in 2026. |
| Important source limitations | The available fields describe places and coordinates only; live event, booking, price, opening-hour, accessibility, and availability information are not available from the source. |

In [ ]:
provenance_validation = pd.DataFrame({
    "Check": [
        "Raw dataframe structure",
        "Processed dataframe structure",
        "Original raw columns retained in processed dataframe"
    ],
    "Result": [
        f"{df.shape[0]} rows x {df.shape[1]} columns",
        f"{prepared_df.shape[0]} rows x {prepared_df.shape[1]} columns",
        set(df.columns).issubset(prepared_df.columns)
    ]
})

print(provenance_validation.to_string(index=False))

## 3.3 Data Limitations

| Limitation supported by the dataset inspection | Why it matters for AgeTogether's trustworthy local discovery feature |
|---|---|
| The dataset contains places of interest, not confirmed live events. | A listed place must not be presented as proof that an activity is currently occurring. |
| The official dataset was modified on 12 March 2021, and its last recorded data processing date is 13 November 2022. The team acquired its local CSV copy on 1 September 2026, which does not make the data current in 2026. | Freshness is a limitation for a 2026 prototype. AgeTogether should not claim that place details are current or real-time without another validated source. |
| Event dates and times are not available from the source. | The feature cannot state when an event takes place. |
| Price information is not available from the source. | The feature cannot state a cost or describe a place as free. |
| Booking availability is not available from the source. | The feature cannot state that a user can book, reserve, or attend. |
| Explicit accessibility information is not available from the source. | The feature cannot make claims about mobility access or age-friendly facilities. |
| Opening hours are not available from the source. | The feature cannot state when a place is open. |
| Coordinates are available but were originally stored together as one text field. | They require a documented split and numeric conversion before location-based analysis. |
| No obvious unique record identifier exists in the four original columns. | Records cannot be safely deduplicated using an identifier. |
| Repeated Feature Names have different coordinate pairs in this dataset. | A repeated name should not automatically be removed as a duplicate record. |
| The source does not provide event dates/times, prices, bookings, opening hours, explicit accessibility information, or live availability. | AgeTogether must not present any of these attributes as confirmed facts unless another validated source is introduced. |

# 4. Relevance Analysis

This section inspects the dataset structure to prepare evidence for a later product-relevance decision. It does not filter, remove, rank, or recommend any records.

## 4.1 Sub Theme Inventory

The table below lists every Sub Theme, its corresponding source Theme where available, its record count, and its percentage of all 242 records.

In [ ]:
total_records = len(prepared_df)
sub_theme_counts = prepared_df.groupby("Sub Theme").size().rename("Number of records")
sub_theme_themes = (
    prepared_df.groupby("Sub Theme")["Theme"]
    .agg(lambda values: " | ".join(sorted(values.unique())))
    .rename("Theme")
)
sub_theme_inventory = (
    pd.concat([sub_theme_themes, sub_theme_counts], axis=1)
    .reset_index()
    .assign(**{"Percentage of total records": lambda table: (table["Number of records"] / total_records * 100).round(2)})
    .sort_values(["Number of records", "Sub Theme"], ascending=[False, True])
    .reset_index(drop=True)
)

print(sub_theme_inventory.to_string(index=False))

## 4.2 Theme Distribution

This table shows the distribution of records across the source Theme categories.

In [ ]:
theme_distribution = (
    prepared_df.groupby("Theme")
    .size()
    .rename("Number of records")
    .reset_index()
    .assign(**{"Percentage of total records": lambda table: (table["Number of records"] / total_records * 100).round(2)})
    .sort_values(["Number of records", "Theme"], ascending=[False, True])
    .reset_index(drop=True)
)

print(theme_distribution.to_string(index=False))
print("Theme record-count total:", theme_distribution["Number of records"].sum())
print("Sub Theme record-count total:", sub_theme_inventory["Number of records"].sum())

**Note:** The purpose of this step is to understand the dataset structure before defining product-relevance rules. No records have been removed.

## 4.3 Product-Relevance Classification

We preserve all source records and add a product-relevance layer. The classification is based on the purpose of AgeTogether's Iteration 1 local discovery feature, not on whether a place is generally valuable. Tier 1 means direct discovery relevance; Tier 2 means supporting or access relevance; Tier 3 means outside the current discovery purpose. Ambiguous categories are marked Needs Review rather than silently excluded.

This dataset represents places of interest, not confirmed live activities. Classification does not establish that an event occurs at a place. No event dates, prices, bookings, opening hours, accessibility information, or availability are inferred.

In [ ]:
tier_1_sub_themes = {
    "Informal Outdoor Facility (Park/Garden/Reserve)",
    "Major Sports & Recreation Facility",
    "Indoor Recreation Facility",
    "Outdoor Recreation Facility (Zoo, Golf Course)",
    "Gymnasium/Health Club",
    "Private Sports Club/Facility",
    "Observation Tower/Wheel",
    "Art Gallery/Museum",
    "Theatre Live",
    "Cinema",
    "Aquarium",
    "Library",
    "Visitor Centre",
    "Function/Conference/Exhibition Centre",
    "Marina"
}

tier_2_sub_themes = {
    "Railway Station",
    "Transport Terminal",
    "Bridge",
    "Church",
    "Synagogue",
    "Public Hospital",
    "Private Hospital",
    "Medical Services",
    "Primary Schools",
    "Secondary Schools",
    "School - Primary and Secondary Education",
    "Tertiary (University)",
    "Further Education",
    "Government Building"
}

tier_3_sub_themes = {
    "Office",
    "Retail",
    "Retail/Office",
    "Retail/Office/Carpark",
    "Retail/Office/Residential/Carpark",
    "Retail/Residential",
    "Industrial (Manufacturing)",
    "Vacant Land - Undeveloped Site",
    "Current Construction Site",
    "Current Construction Site - Commercial",
    "Dwelling (House)",
    "Hostel",
    "Police Station",
    "Fire Station",
    "Store Yard"
}

ambiguous_sub_themes = {"Public Buildings"}
actual_sub_themes = set(prepared_df["Sub Theme"].unique())
covered_sub_themes = tier_1_sub_themes | tier_2_sub_themes | tier_3_sub_themes | ambiguous_sub_themes
uncovered_sub_themes = sorted(actual_sub_themes - covered_sub_themes)

def classify_sub_theme(sub_theme):
    if sub_theme in tier_1_sub_themes:
        return "Tier 1 - Discovery Place", "Supports direct local discovery through recreation, cultural participation, or community participation."
    if sub_theme in tier_2_sub_themes:
        return "Tier 2 - Supporting/Access Place", "Supports access, attendance planning, or local context; it is not an automatic activity result."
    if sub_theme in tier_3_sub_themes:
        return "Tier 3 - Not Relevant", "Outside the current Iteration 1 local discovery purpose."
    if sub_theme in ambiguous_sub_themes:
        return "Needs Review", "This Sub Theme contains mixed record contexts, including cultural venues and courts, so it needs a team-approved rule."
    return "Needs Review", "No proposed product-relevance rule covers this Sub Theme."

classified_df = prepared_df.copy()
classified_df[["relevance_tier", "relevance_reason"]] = (
    classified_df["Sub Theme"].apply(classify_sub_theme).apply(pd.Series)
)

coverage_review = pd.DataFrame({
    "Check": ["Uncovered Sub Themes", "Ambiguous Sub Themes requiring review"],
    "Result": [", ".join(uncovered_sub_themes) if uncovered_sub_themes else "None", ", ".join(sorted(ambiguous_sub_themes))]
})

print(coverage_review.to_string(index=False))

### Classification Summary Evidence

The following tables show the classification results while retaining every source record. Needs Review is used for uncovered or contextually mixed categories.

In [ ]:
tier_order = [
    "Tier 1 - Discovery Place",
    "Tier 2 - Supporting/Access Place",
    "Tier 3 - Not Relevant",
    "Needs Review"
]
tier_summary = (
    classified_df["relevance_tier"]
    .value_counts()
    .reindex(tier_order, fill_value=0)
    .rename_axis("relevance_tier")
    .reset_index(name="Number of records")
)
tier_summary["Percentage of all records"] = (
    tier_summary["Number of records"] / len(classified_df) * 100
).round(2)

sub_theme_relevance_summary = (
    classified_df.groupby(["Sub Theme", "Theme", "relevance_tier"], as_index=False)
    .size()
    .rename(columns={"size": "Number of records"})
)
sub_theme_relevance_summary["relevance_tier"] = pd.Categorical(
    sub_theme_relevance_summary["relevance_tier"],
    categories=tier_order,
    ordered=True
)
sub_theme_relevance_summary = (
    sub_theme_relevance_summary
    .sort_values(["relevance_tier", "Number of records", "Sub Theme"], ascending=[True, False, True])
    .reset_index(drop=True)
)

print(tier_summary.to_string(index=False))
print(sub_theme_relevance_summary.to_string(index=False))
print("Classification record-count total:", tier_summary["Number of records"].sum())

## 4.4 Needs Review Investigation

The following investigation examines every record currently marked Needs Review. It does not change any existing product-relevance classification.

In [ ]:
needs_review_records = (
    classified_df.loc[
        classified_df["relevance_tier"] == "Needs Review",
        ["Feature Name", "Theme", "Sub Theme", "latitude", "longitude"]
    ]
    .sort_values(["Sub Theme", "Feature Name"])
    .reset_index(drop=True)
)

print(needs_review_records.to_string(index=False))

In [ ]:
needs_review_group_summary = (
    needs_review_records.groupby("Sub Theme", as_index=False)
    .agg(
        **{
            "Number of records": ("Feature Name", "size"),
            "Feature Names": ("Feature Name", lambda values: " | ".join(sorted(values)))
        }
    )
    .sort_values(["Sub Theme"])
    .reset_index(drop=True)
)

print(needs_review_group_summary.to_string(index=False))

### Public Buildings Investigation

Public Buildings is too broad to classify reliably at the Sub Theme level because the records include both cultural/community destinations and institutional buildings. Further review at the Feature Name level is therefore required.

In [ ]:
public_buildings_records = (
    needs_review_records.loc[needs_review_records["Sub Theme"] == "Public Buildings", ["Feature Name"]]
    .sort_values("Feature Name")
    .reset_index(drop=True)
)

print(public_buildings_records.to_string(index=False))

### Other Needs Review Categories

Casino, Cemetery, Department Store, and Film & RV Studio are not covered by the current rule set. Their product relevance cannot be determined from the current rule set without a product-scope decision.

In [ ]:
other_needs_review_sub_themes = {"Casino", "Cemetery", "Department Store", "Film & RV Studio"}
other_needs_review_records = (
    needs_review_records.loc[
        needs_review_records["Sub Theme"].isin(other_needs_review_sub_themes),
        ["Sub Theme", "Feature Name"]
    ]
    .sort_values(["Sub Theme", "Feature Name"])
    .reset_index(drop=True)
)

print(other_needs_review_records.to_string(index=False))

### Suggested Review Directions

The suggestions below are based only on the Feature Names in this dataset and AgeTogether's current local discovery purpose. They are not final decisions and do not change `relevance_tier`. A Feature Name alone cannot confirm whether an activity occurs, whether the public can attend, or whether a place is accessible.

In [ ]:
suggested_review_tiers = {
    "Commonwealth Law Courts": "Tier 3 review",
    "Conservatory": "Tier 1 review",
    "County Court Melbourne": "Tier 3 review",
    "Elisabeth Murdoch Hall": "Tier 1 review",
    "Melbourne Childrens Court": "Tier 3 review",
    "Melbourne Recital Centre": "Tier 1 review",
    "Melbourne Theatre Company": "Tier 1 review",
    "Melbourne Town Hall": "Tier 2 review",
    "NGV International": "Tier 1 review",
    "Shrine of Remembrance": "Tier 1 review",
    "Sidney Myer Music Bowl": "Tier 1 review",
    "State Library Victoria": "Tier 1 review",
    "Supreme Court": "Tier 3 review",
    "Crown Entertainment Complex": "Tier 1 review",
    "Melbourne General Cemetery": "Tier 2 review",
    "David Jones": "Tier 3 review",
    "Myer": "Tier 3 review",
    "Central City Studios": "Tier 3 review",
    "Channel 7 - Melbourne Broadcast Centre": "Tier 3 review"
}

suggested_review_reasons = {
    "Commonwealth Law Courts": "The Feature Name identifies a court, which is outside the current local discovery purpose.",
    "Conservatory": "The Feature Name may indicate a cultural destination, so it warrants Tier 1 review.",
    "County Court Melbourne": "The Feature Name identifies a court, which is outside the current local discovery purpose.",
    "Elisabeth Murdoch Hall": "The Feature Name identifies a hall, which may support cultural participation and warrants Tier 1 review.",
    "Melbourne Childrens Court": "The Feature Name identifies a court, which is outside the current local discovery purpose.",
    "Melbourne Recital Centre": "The Feature Name indicates a recital centre, which may support cultural participation and warrants Tier 1 review.",
    "Melbourne Theatre Company": "The Feature Name indicates a theatre organisation, which may support cultural participation and warrants Tier 1 review.",
    "Melbourne Town Hall": "The Feature Name identifies a civic building; a Tier 2 review is appropriate because it may provide community context but does not confirm an activity.",
    "NGV International": "The Feature Name indicates a gallery destination, which may support cultural participation and warrants Tier 1 review.",
    "Shrine of Remembrance": "The Feature Name indicates a cultural or heritage destination, so it warrants Tier 1 review.",
    "Sidney Myer Music Bowl": "The Feature Name indicates a music venue, which may support cultural participation and warrants Tier 1 review.",
    "State Library Victoria": "The Feature Name indicates a library destination, which may support local discovery and warrants Tier 1 review.",
    "Supreme Court": "The Feature Name identifies a court, which is outside the current local discovery purpose.",
    "Crown Entertainment Complex": "The Feature Name indicates an entertainment complex, which may support local discovery and warrants Tier 1 review; it does not confirm a live event.",
    "Melbourne General Cemetery": "The Feature Name identifies a cemetery; a Tier 2 review is appropriate for local context, not as an automatic activity result.",
    "David Jones": "The Feature Name identifies a department store, which is outside the current discovery purpose.",
    "Myer": "The Feature Name identifies a department store, which is outside the current discovery purpose.",
    "Central City Studios": "The Feature Name identifies a studio, but the source does not establish public discovery relevance.",
    "Channel 7 - Melbourne Broadcast Centre": "The Feature Name identifies a broadcast centre, but the source does not establish public discovery relevance."
}

needs_review_suggestions = needs_review_records.copy()
needs_review_suggestions["Suggested review direction"] = needs_review_suggestions["Feature Name"].map(suggested_review_tiers)
needs_review_suggestions["Reason"] = needs_review_suggestions["Feature Name"].map(suggested_review_reasons)

print(needs_review_suggestions.to_string(index=False))

## 4.5 Final Product-Relevance Classification

We first classified records at the Sub Theme level. Where a Sub Theme was too broad or not covered by the initial rules, we reviewed the Feature Name. This allowed us to refine ambiguous records without deleting source data.

The classification is a product-relevance decision, not a claim that a place hosts a live activity. It does not infer event dates, prices, bookings, opening hours, accessibility information, or availability. Raw source records are preserved.

In [ ]:
final_decisions = {
    "Conservatory": ("Tier 1 - Discovery Place", "Identifiable cultural, recreational, educational, or public destination that can reasonably support local discovery; no live activity is inferred."),
    "Elisabeth Murdoch Hall": ("Tier 1 - Discovery Place", "Identifiable cultural, recreational, educational, or public destination that can reasonably support local discovery; no live activity is inferred."),
    "Melbourne Recital Centre": ("Tier 1 - Discovery Place", "Identifiable cultural, recreational, educational, or public destination that can reasonably support local discovery; no live activity is inferred."),
    "Melbourne Theatre Company": ("Tier 1 - Discovery Place", "Identifiable cultural, recreational, educational, or public destination that can reasonably support local discovery; no live activity is inferred."),
    "NGV International": ("Tier 1 - Discovery Place", "Identifiable cultural, recreational, educational, or public destination that can reasonably support local discovery; no live activity is inferred."),
    "Shrine of Remembrance": ("Tier 1 - Discovery Place", "Identifiable cultural, recreational, educational, or public destination that can reasonably support local discovery; no live activity is inferred."),
    "Sidney Myer Music Bowl": ("Tier 1 - Discovery Place", "Identifiable cultural, recreational, educational, or public destination that can reasonably support local discovery; no live activity is inferred."),
    "State Library Victoria": ("Tier 1 - Discovery Place", "Identifiable cultural, recreational, educational, or public destination that can reasonably support local discovery; no live activity is inferred."),
    "Crown Entertainment Complex": ("Tier 2 - Supporting/Access Place", "May provide local, community, or cultural context, but the source identifies it as a Casino and does not establish current AgeTogether-suitable activities."),
    "Melbourne General Cemetery": ("Tier 2 - Supporting/Access Place", "May have supporting cultural or local relevance, but it is not classified as an activity."),
    "Melbourne Town Hall": ("Tier 2 - Supporting/Access Place", "May provide local or community context, but the source does not establish a current activity."),
    "David Jones": ("Tier 3 - Not Relevant", "Does not directly support the current AgeTogether local discovery purpose based on the available dataset information."),
    "Myer": ("Tier 3 - Not Relevant", "Does not directly support the current AgeTogether local discovery purpose based on the available dataset information."),
    "Central City Studios": ("Tier 3 - Not Relevant", "Does not directly support the current AgeTogether local discovery purpose based on the available dataset information."),
    "Channel 7 - Melbourne Broadcast Centre": ("Tier 3 - Not Relevant", "Does not directly support the current AgeTogether local discovery purpose based on the available dataset information."),
    "Commonwealth Law Courts": ("Tier 3 - Not Relevant", "Does not directly support the current AgeTogether local discovery purpose based on the available dataset information."),
    "County Court Melbourne": ("Tier 3 - Not Relevant", "Does not directly support the current AgeTogether local discovery purpose based on the available dataset information."),
    "Melbourne Childrens Court": ("Tier 3 - Not Relevant", "Does not directly support the current AgeTogether local discovery purpose based on the available dataset information."),
    "Supreme Court": ("Tier 3 - Not Relevant", "Does not directly support the current AgeTogether local discovery purpose based on the available dataset information.")
}

final_classified_df = classified_df.copy()
final_classified_df["final_relevance_tier"] = final_classified_df["relevance_tier"]
final_classified_df["final_relevance_reason"] = final_classified_df["relevance_reason"]

needs_review_mask = final_classified_df["relevance_tier"] == "Needs Review"
final_classified_df.loc[needs_review_mask, "final_relevance_tier"] = (
    final_classified_df.loc[needs_review_mask, "Feature Name"]
    .map(lambda name: final_decisions.get(name, (None, None))[0])
)
final_classified_df.loc[needs_review_mask, "final_relevance_reason"] = (
    final_classified_df.loc[needs_review_mask, "Feature Name"]
    .map(lambda name: final_decisions.get(name, (None, None))[1])
)

unresolved_final_records = final_classified_df.loc[
    final_classified_df["final_relevance_tier"].isna(),
    ["Feature Name", "Sub Theme"]
]

print("Final classification unresolved records:", len(unresolved_final_records))
if not unresolved_final_records.empty:
    print(unresolved_final_records.to_string(index=False))

In [ ]:
final_tier_order = [
    "Tier 1 - Discovery Place",
    "Tier 2 - Supporting/Access Place",
    "Tier 3 - Not Relevant"
]
final_tier_summary = (
    final_classified_df["final_relevance_tier"]
    .value_counts()
    .reindex(final_tier_order, fill_value=0)
    .rename_axis("final_relevance_tier")
    .reset_index(name="Number of records")
)
final_tier_summary["Percentage of all records"] = (
    final_tier_summary["Number of records"] / len(final_classified_df) * 100
).round(2)

final_sub_theme_summary = (
    final_classified_df.groupby(["Sub Theme", "final_relevance_tier"], as_index=False)
    .size()
    .rename(columns={"size": "Number of records"})
)
final_sub_theme_summary["final_relevance_tier"] = pd.Categorical(
    final_sub_theme_summary["final_relevance_tier"],
    categories=final_tier_order,
    ordered=True
)
final_sub_theme_summary = (
    final_sub_theme_summary
    .sort_values(["final_relevance_tier", "Number of records", "Sub Theme"], ascending=[True, False, True])
    .reset_index(drop=True)
)

finalised_needs_review_records = (
    final_classified_df.loc[needs_review_mask, ["Feature Name", "Sub Theme", "final_relevance_tier", "final_relevance_reason"]]
    .sort_values(["final_relevance_tier", "Feature Name"])
    .reset_index(drop=True)
)

print(final_tier_summary.to_string(index=False))
print(final_sub_theme_summary.to_string(index=False))
print(finalised_needs_review_records.to_string(index=False))
print("Final classification record-count total:", final_tier_summary["Number of records"].sum())

## 4.6 Product Context

AgeTogether Iteration 1 focuses on trusted local discovery. The final product-relevance layer supports later product-scope work, but it does not state that a listed place is hosting an activity or is currently available.

# 5. Descriptive Analysis

The purpose of this analysis is to understand the composition and geographic coverage of the places classified as relevant to AgeTogether's Iteration 1 local discovery feature. This analysis describes the dataset; it does not establish popularity, suitability, accessibility, or live availability.

## 5.1 Final Tier Distribution

In [ ]:
descriptive_final_tier_distribution = (
    final_classified_df["final_relevance_tier"]
    .value_counts()
    .reindex(final_tier_order, fill_value=0)
    .rename_axis("final_relevance_tier")
    .reset_index(name="Count")
)
descriptive_final_tier_distribution["Percentage"] = (
    descriptive_final_tier_distribution["Count"] / len(final_classified_df) * 100
).round(2)

print(descriptive_final_tier_distribution.to_string(index=False))

## 5.2 Tier 1 Discovery Place Composition

In [ ]:
tier_1_df = final_classified_df.loc[
    final_classified_df["final_relevance_tier"] == "Tier 1 - Discovery Place"
].copy()
tier_1_total = len(tier_1_df)
tier_1_sub_theme_distribution = (
    tier_1_df.groupby(["Sub Theme", "Theme"], as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .assign(**{"Percentage of Tier 1": lambda table: (table["Count"] / tier_1_total * 100).round(2)})
    .sort_values(["Count", "Sub Theme"], ascending=[False, True])
    .reset_index(drop=True)
)

print(tier_1_sub_theme_distribution.to_string(index=False))

## 5.3 Tier 1 Theme Distribution

In [ ]:
tier_1_theme_distribution = (
    tier_1_df.groupby("Theme")
    .size()
    .rename("Count")
    .reset_index()
    .assign(**{"Percentage of Tier 1": lambda table: (table["Count"] / tier_1_total * 100).round(2)})
    .sort_values(["Count", "Theme"], ascending=[False, True])
    .reset_index(drop=True)
)

print(tier_1_theme_distribution.to_string(index=False))

## 5.4 Category Coverage

Category coverage describes the number and spread of source categories represented in Tier 1. It does not measure popularity or user preference.

In [ ]:
most_common_5_sub_themes = tier_1_sub_theme_distribution.head(5)
least_common_count = tier_1_sub_theme_distribution["Count"].min()
least_common_sub_themes = tier_1_sub_theme_distribution.loc[
    tier_1_sub_theme_distribution["Count"] == least_common_count
].sort_values("Sub Theme")

category_coverage_summary = pd.DataFrame({
    "Measure": ["Unique Themes represented in Tier 1", "Unique Sub Themes represented in Tier 1"],
    "Result": [tier_1_df["Theme"].nunique(), tier_1_df["Sub Theme"].nunique()]
})

print(category_coverage_summary.to_string(index=False))
print("Most common 5 Sub Themes:")
print(most_common_5_sub_themes.to_string(index=False))
print("Least common Sub Themes:")
print(least_common_sub_themes.to_string(index=False))

## 5.5 Geographic Coverage

This check reports the coordinate range of Tier 1 records. It describes the geographic spread of the supplied coordinates only; it does not establish access, distance from a user, or service coverage.

In [ ]:
tier_1_valid_coordinates = tier_1_df.loc[
    tier_1_df["latitude"].between(-90, 90)
    & tier_1_df["longitude"].between(-180, 180)
].copy()

geographic_coverage_summary = pd.DataFrame({
    "Measure": [
        "Minimum latitude",
        "Maximum latitude",
        "Minimum longitude",
        "Maximum longitude",
        "Tier 1 places with valid coordinates"
    ],
    "Result": [
        tier_1_valid_coordinates["latitude"].min(),
        tier_1_valid_coordinates["latitude"].max(),
        tier_1_valid_coordinates["longitude"].min(),
        tier_1_valid_coordinates["longitude"].max(),
        len(tier_1_valid_coordinates)
    ]
})

print(geographic_coverage_summary.to_string(index=False))

## 5.6 Initial Findings

- Tier 1 contains 115 records. Leisure/Recreation is the largest Theme with 63 records (54.78% of Tier 1), followed by Place Of Assembly with 40 records (34.78%). Together, these two Themes account for 103 Tier 1 records (89.57%).
- Tier 1 includes 5 Themes and 16 Sub Themes, showing category variety within the selected product-relevance layer. However, the record distribution is uneven: Informal Outdoor Facility (Park/Garden/Reserve) is the largest Sub Theme with 37 records (32.17%), while five Sub Themes have one record each.
- All 115 Tier 1 records have valid coordinates. Their latitude range is -37.841400 to -37.781917 and their longitude range is 144.908825 to 144.989063. This describes the spread of the supplied locations only.
- These counts and coordinate ranges do not demonstrate popularity, quality, safety, accessibility, user preference, or current activity availability. The category balance also reflects the source dataset and the product-relevance rules, not demand from older adults.

# 6. Product Data View and Export

This step creates non-destructive output datasets from the final classified data for product integration and technical evidence. The raw CSV remains unchanged, and no ranking or inferred factual attributes are added.

The complete classified export is retained as DS evidence and preserves all source records. The discovery export contains only places classified as directly relevant to the current AgeTogether local discovery feature. These records are places of interest, not confirmed live activities.

The output must not be interpreted as confirming event dates, prices, opening hours, accessibility, bookings, suitability, or live availability. No ranking has been applied.

In [ ]:
from pathlib import Path

product_classified_columns = [
    "Feature Name",
    "Theme",
    "Sub Theme",
    "latitude",
    "longitude",
    "final_relevance_tier",
    "final_relevance_reason"
]
product_classified_df = final_classified_df[product_classified_columns].copy()

discovery_places_columns = [
    "Feature Name",
    "Theme",
    "Sub Theme",
    "latitude",
    "longitude"
]
discovery_places_df = final_classified_df.loc[
    final_classified_df["final_relevance_tier"] == "Tier 1 - Discovery Place",
    discovery_places_columns
].copy()
discovery_places_df["source_dataset"] = "City of Melbourne - Landmarks and places of interest"

print("Product classified dataframe records:", len(product_classified_df))
print("Discovery places dataframe records:", len(discovery_places_df))
print("Product classified columns:", product_classified_df.columns.tolist())
print("Discovery places columns:", discovery_places_df.columns.tolist())

In [ ]:
output_directory = Path("../data/processed")
if not output_directory.exists():
    output_directory = Path("data/processed")
output_directory.mkdir(parents=True, exist_ok=True)
product_classified_export_path = output_directory / "agetogether_places_classified.csv"
discovery_places_export_path = output_directory / "agetogether_discovery_places.csv"

product_classified_df.to_csv(product_classified_export_path, index=False)
discovery_places_df.to_csv(discovery_places_export_path, index=False)

print("Classified export path:", product_classified_export_path.resolve())
print("Discovery export path:", discovery_places_export_path.resolve())


In [ ]:
exported_product_classified_df = pd.read_csv(product_classified_export_path)
exported_discovery_places_df = pd.read_csv(discovery_places_export_path)
ranking_field_terms = {"rank", "score", "recommendation", "popularity"}
export_column_text = " ".join(exported_product_classified_df.columns.str.lower()) + " " + " ".join(exported_discovery_places_df.columns.str.lower())
ranking_fields_present = any(term in export_column_text for term in ranking_field_terms)

export_validation = pd.DataFrame({
    "Check": [
        "Source dataframe records",
        "Product classified dataframe records",
        "Discovery places dataframe records",
        "Exported classified CSV records",
        "Exported discovery CSV records",
        "Ranking fields present in exports",
        "Raw source dataframe structure"
    ],
    "Result": [
        len(df),
        len(product_classified_df),
        len(discovery_places_df),
        len(exported_product_classified_df),
        len(exported_discovery_places_df),
        ranking_fields_present,
        f"{df.shape[0]} rows x {df.shape[1]} columns"
    ]
})

print(export_validation.to_string(index=False))

# 7. Transparent Rule-Based Ranking Baseline

This section demonstrates an explainable Iteration 1 ranking baseline using only source-supported place category and coordinate attributes. It is not machine learning, a claim of true user preference, a popularity ranking, or a prediction of suitability for older adults. It is a transparent prototype mechanism that can later be evaluated and refined using explicit user feedback.

The ranking uses a fixed demonstration scenario only. It does not create recommendations for real people.


## 7.1 Demonstration Scenario and Ranking Rules

**Demonstration user location:** latitude **-37.8136**, longitude **144.9631**. This is a fixed example location near central Melbourne; it is not a real user's location.

The fixed demonstration preferences are ranked as follows:

1. Informal Outdoor Facility (Park/Garden/Reserve)
2. Art Gallery/Museum
3. Theatre Live

Category relevance rule: preference rank 1 receives `category_score = 3`; rank 2 receives `category_score = 2`; rank 3 receives `category_score = 1`; all other Tier 1 places receive `category_score = 0`.

Distance rule: `distance_score = 3` for distances at or below 1 km; `2` for more than 1 km and at or below 3 km; `1` for more than 3 km and at or below 5 km; and `0` for more than 5 km. These pre-defined bands are retained because they provide a simple, visible baseline.

`ranking_score = category_score + distance_score`. Results are sorted by ranking score descending, category score descending, straight-line distance ascending, then Feature Name ascending. This final alphabetical tie-break makes identical-score results deterministic (repeatable).


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ranking_input_path = Path("../data/processed/agetogether_discovery_places.csv")
if not ranking_input_path.exists():
    ranking_input_path = Path("data/processed/agetogether_discovery_places.csv")

ranking_input_df = pd.read_csv(ranking_input_path)

demonstration_user_location = {
    "latitude": -37.8136,
    "longitude": 144.9631
}

preferred_subthemes = [
    "Informal Outdoor Facility (Park/Garden/Reserve)",
    "Art Gallery/Museum",
    "Theatre Live"
]

category_score_rules = {
    preferred_subthemes[0]: 3,
    preferred_subthemes[1]: 2,
    preferred_subthemes[2]: 1
}

print("Input records:", len(ranking_input_df))
print("Demonstration user location:", demonstration_user_location)
print("Demonstration preferred Sub Themes:", preferred_subthemes)


## 7.2 Distance Calculation

Straight-line geographic distance is calculated using the Haversine formula. `distance_km` is not walking distance or travel time, and does not account for roads, accessibility, public transport, safety, or current conditions.


In [ ]:
def haversine_distance_km(latitude, longitude, reference_latitude, reference_longitude):
    earth_radius_km = 6371.0
    latitude_radians = np.radians(latitude)
    longitude_radians = np.radians(longitude)
    reference_latitude_radians = np.radians(reference_latitude)
    reference_longitude_radians = np.radians(reference_longitude)
    delta_latitude = reference_latitude_radians - latitude_radians
    delta_longitude = reference_longitude_radians - longitude_radians
    haversine_value = (
        np.sin(delta_latitude / 2) ** 2
        + np.cos(latitude_radians)
        * np.cos(reference_latitude_radians)
        * np.sin(delta_longitude / 2) ** 2
    )
    return 2 * earth_radius_km * np.arcsin(np.sqrt(haversine_value))

ranking_demo_df = ranking_input_df.copy()
ranking_demo_df["distance_km"] = haversine_distance_km(
    ranking_demo_df["latitude"],
    ranking_demo_df["longitude"],
    demonstration_user_location["latitude"],
    demonstration_user_location["longitude"]
)
ranking_demo_df["category_score"] = ranking_demo_df["Sub Theme"].map(category_score_rules).fillna(0).astype(int)
ranking_demo_df["distance_score"] = np.select(
    [
        ranking_demo_df["distance_km"] <= 1,
        ranking_demo_df["distance_km"] <= 3,
        ranking_demo_df["distance_km"] <= 5
    ],
    [3, 2, 1],
    default=0
).astype(int)
ranking_demo_df["ranking_score"] = ranking_demo_df["category_score"] + ranking_demo_df["distance_score"]

def create_ranking_reason(row):
    if row["distance_km"] <= 1:
        distance_text = "within 1 km of the demonstration location"
    else:
        distance_text = f"approximately {row['distance_km']:.1f} km straight-line distance from the demonstration location"
    if row["category_score"] > 0:
        preference_rank = 4 - row["category_score"]
        return f"Matches demonstration preferred category rank {preference_rank}: {row['Sub Theme']}; {distance_text}."
    return f"No demonstration preferred-category match; ranked mainly by geographic proximity ({distance_text})."

ranking_demo_df["ranking_reason"] = ranking_demo_df.apply(create_ranking_reason, axis=1)
ranking_demo_df = ranking_demo_df.sort_values(
    by=["ranking_score", "category_score", "distance_km", "Feature Name"],
    ascending=[False, False, True, True],
    kind="mergesort"
).reset_index(drop=True)

top_10_ranked_places = ranking_demo_df.head(10)
score_distribution = ranking_demo_df["ranking_score"].value_counts().sort_index(ascending=False).rename_axis("ranking_score").reset_index(name="place_count")
category_score_distribution = ranking_demo_df["category_score"].value_counts().sort_index(ascending=False).rename_axis("category_score").reset_index(name="place_count")
distance_score_distribution = ranking_demo_df["distance_score"].value_counts().sort_index(ascending=False).rename_axis("distance_score").reset_index(name="place_count")

display_columns = ["Feature Name", "Theme", "Sub Theme", "distance_km", "category_score", "distance_score", "ranking_score", "ranking_reason"]
print("Top 10 ranked places")
display(top_10_ranked_places[display_columns])
print("Score distribution")
display(score_distribution)
print("Category-score distribution")
display(category_score_distribution)
print("Distance-score distribution")
display(distance_score_distribution)


## 7.3 Ranking Evidence and Interpretation

The top 10 table, score distribution, category-score distribution, and distance-score distribution below explain why places rank highly. A high result is caused only by the stated preferred Sub Theme rule, the stated straight-line distance band, or both. It does not imply popularity, quality, accessibility, safety, live availability, or suitability for older adults.


## 7.4 Sensitivity and Transparency Check

This check changes one visible assumption only: it swaps the first two preferred Sub Theme positions. The comparison shows whether the top results change. Rule-based systems depend on the chosen rules; future iterations should validate category choices and distance bands with explicit user feedback rather than treating this demonstration scenario as a user-preference model.


In [ ]:
sensitivity_preferred_subthemes = [
    "Art Gallery/Museum",
    "Informal Outdoor Facility (Park/Garden/Reserve)",
    "Theatre Live"
]
sensitivity_category_score_rules = {
    sensitivity_preferred_subthemes[0]: 3,
    sensitivity_preferred_subthemes[1]: 2,
    sensitivity_preferred_subthemes[2]: 1
}

sensitivity_df = ranking_input_df.copy()
sensitivity_df["distance_km"] = haversine_distance_km(
    sensitivity_df["latitude"],
    sensitivity_df["longitude"],
    demonstration_user_location["latitude"],
    demonstration_user_location["longitude"]
)
sensitivity_df["category_score"] = sensitivity_df["Sub Theme"].map(sensitivity_category_score_rules).fillna(0).astype(int)
sensitivity_df["distance_score"] = np.select(
    [
        sensitivity_df["distance_km"] <= 1,
        sensitivity_df["distance_km"] <= 3,
        sensitivity_df["distance_km"] <= 5
    ],
    [3, 2, 1],
    default=0
).astype(int)
sensitivity_df["ranking_score"] = sensitivity_df["category_score"] + sensitivity_df["distance_score"]
sensitivity_df = sensitivity_df.sort_values(
    by=["ranking_score", "category_score", "distance_km", "Feature Name"],
    ascending=[False, False, True, True],
    kind="mergesort"
).reset_index(drop=True)

baseline_top_10_names = ranking_demo_df.head(10)["Feature Name"].tolist()
sensitivity_top_10_names = sensitivity_df.head(10)["Feature Name"].tolist()
sensitivity_comparison = pd.DataFrame({
    "baseline_top_10": pd.Series(baseline_top_10_names),
    "swapped_preference_top_10": pd.Series(sensitivity_top_10_names)
})

print("Sensitivity change: Art Gallery/Museum is rank 1 and Informal Outdoor Facility is rank 2.")
print("Top 10 changes:", baseline_top_10_names != sensitivity_top_10_names)
display(sensitivity_comparison)


## 7.5 Demonstration Export and Validation

The CSV export is ranking evidence for this fixed demonstration scenario. It preserves all 115 input places and includes only source-supported fields plus documented, derived ranking fields. It must not be treated as a live recommendation feed or as a statement about any real user.


In [ ]:
ranking_export_columns = [
    "Feature Name",
    "Theme",
    "Sub Theme",
    "latitude",
    "longitude",
    "distance_km",
    "category_score",
    "distance_score",
    "ranking_score",
    "ranking_reason",
    "source_dataset"
]
ranking_export_df = ranking_demo_df[ranking_export_columns].copy()
ranking_export_df["ranking_context"] = "Demonstration ranking baseline only; not a live recommendation."
ranking_export_path = ranking_input_path.parent / "agetogether_ranking_baseline_demo.csv"
ranking_export_df.to_csv(ranking_export_path, index=False)

expected_distance_score = np.select(
    [
        ranking_demo_df["distance_km"] <= 1,
        ranking_demo_df["distance_km"] <= 3,
        ranking_demo_df["distance_km"] <= 5
    ],
    [3, 2, 1],
    default=0
).astype(int)
re_sorted_df = ranking_demo_df.sort_values(
    by=["ranking_score", "category_score", "distance_km", "Feature Name"],
    ascending=[False, False, True, True],
    kind="mergesort"
).reset_index(drop=True)
unsupported_export_fields = {
    "event_date", "event_time", "price", "booking", "opening_hours",
    "accessibility", "live_availability", "popularity", "recommendation_score"
}

ranking_validation = pd.DataFrame({
    "Check": [
        "Input records",
        "Ranked records",
        "Missing latitude or longitude",
        "Ranking score follows category_score + distance_score rule",
        "Distance score follows documented bands",
        "Ranking ordering is deterministic",
        "Unsupported factual fields in export",
        "Exported records"
    ],
    "Result": [
        len(ranking_input_df),
        len(ranking_demo_df),
        int(ranking_demo_df[["latitude", "longitude"]].isna().any(axis=1).sum()),
        bool((ranking_demo_df["ranking_score"] == ranking_demo_df["category_score"] + ranking_demo_df["distance_score"]).all()),
        bool((ranking_demo_df["distance_score"].to_numpy() == expected_distance_score).all()),
        bool(ranking_demo_df["Feature Name"].tolist() == re_sorted_df["Feature Name"].tolist()),
        sorted(set(ranking_export_df.columns) & unsupported_export_fields),
        len(pd.read_csv(ranking_export_path))
    ]
})

print("Ranking export path:", ranking_export_path.resolve())
display(ranking_validation)
